In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import InterContiNetTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

## MNIST

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(
    device="cuda",
    head=head,
    seed=SEED
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model,
    min_acc_limit=0.9,
    paradigm="TIL",
    seed=SEED
)

lr = 0.01
batch_size = 128
epochs = 15
lid_lr = 100
initial_eps = 1
for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        context_id=i,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    interval_trainer.test(test_tasks, context_list=list(range(5)))
    if i == len(train_tasks) - 1:
        continue
    interval_trainer.compute_rashomon_set(
        test, context_id=i, lr=lid_lr, batch_size=256, epochs=1000, default_eps=initial_eps
    )


Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:19<00:00,  1.99s/it, val_loss=0.4856, val_acc=0.9520]


Test Results: [(0.3255, 0.9585), (27.1793, 0.5073), (74.1665, 0.4762), (5.0843, 0.6372), (66.5526, 0.524)] (Avg: (34.6616, 0.6206))
LID size: 6179.0000.


  0%|                                                                                       | 0/1000 [00:00<?, ?it/s, loss=0.3230, acc=0.9688, size=4585.11]


LID size: 4585.1055 with certificate of 0.96875.


...

Test Results: [(0.3255, 0.9585), (0.2189, 0.9834), (0.4281, 0.9672), (0.1578, 0.9848), (66.5519, 0.524)] (Avg: (13.5364, 0.8836))
LID size: 2261.9680.


  0%|                                                                                       | 0/1000 [00:00<?, ?it/s, loss=0.3022, acc=0.9688, size=1123.41]


LID size: 1123.4146 with certificate of 0.96875.


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:18<00:00,  1.83s/it, val_loss=1.0197, val_acc=0.9092]


Test Results: [(0.3255, 0.9585), (0.2189, 0.9834), (0.4281, 0.9672), (0.1578, 0.9848), (0.4823, 0.9456)] (Avg: (0.3225, 0.9679))


### Domain Incremental Learning

In [61]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [64]:
SEED = 2

train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(seed=SEED, device="cuda", output_dim=2)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model,
    min_acc_increment=0.12,
    min_acc_limit=1,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

lr = 0.01
batch_size = 128
epochs = 15
lid_lr = 1000
initial_eps = 1
for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    results = interval_trainer.test(test_tasks)
    print(sum([res[1] for res in results]) / 5)
    if i == len(train_tasks) - 1:
        continue
    target_acc = min(
        max(results[i][1] - interval_trainer.min_acc_increment, results[i][1] / 2),
        interval_trainer.min_acc_limit,
    )
    interval_trainer.min_acc_limit = target_acc
    interval_trainer.compute_rashomon_set(test, lr=lid_lr, default_eps=initial_eps)

Tasks: [[6, 7], [8, 9], [2, 5], [0, 3], [1, 4]]


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 15/15 [00:24<00:00,  1.65s/it, val_loss=0.0912, val_acc=0.9906]


Test Results: [(0.1755, 0.9859), (20.8375, 0.5739), (5.6802, 0.7916), (11.8509, 0.6246), (7.3356, 0.6032)] (Avg: (9.1759, 0.7158))
0.71584
LID size: 1563.0000.


  0%|                                                                                           | 0/500 [00:00<?, ?it/s, loss=0.3665, acc=0.9766, size=0.00]


LID size: 0.0000 with certificate of 0.9765625.


...

Test Results: [(0.1755, 0.9859), (20.8375, 0.5739), (5.6802, 0.7916), (11.8509, 0.6246), (7.3356, 0.6032)] (Avg: (9.1759, 0.7158))
0.71584
LID size: 0.0000.


  0%|                                                                                          | 0/500 [00:00<?, ?it/s, loss=13.5828, acc=0.6172, size=0.00]


LID size: 0.0000 with certificate of 0.6171875.


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 15/15 [00:25<00:00,  1.69s/it, val_loss=8.1451, val_acc=0.6238]


Test Results: [(0.1755, 0.9859), (20.8375, 0.5739), (5.6802, 0.7916), (11.8509, 0.6246), (7.3356, 0.6032)] (Avg: (9.1759, 0.7158))
0.71584


### Class Incremental Training

In [4]:
SEED = 1

train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(
    device="cuda", seed=SEED
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model, min_acc_limit=1, min_acc_increment=0.12, paradigm="CIL", seed=SEED
)

lr = 0.01
batch_size = 128
epochs = 15
lid_lr = 1

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    results = interval_trainer.test(test_tasks)

    if i == len(test_tasks) - 1:
        continue
    target_acc = min(
        max(results[i][1] - interval_trainer.min_acc_increment, results[i][1] / 2),
        interval_trainer.min_acc_limit,
    )
    interval_trainer.min_acc_limit = target_acc
    interval_trainer.compute_rashomon_set(test, lr=lid_lr)


Tasks: [[0, 7], [5, 8], [4, 9], [3, 6], [1, 2]]


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 15/15 [00:41<00:00,  2.75s/it, val_loss=0.1929, val_acc=0.9761]


Test Results: [(0.1803, 0.9736), (8.619, 0.0005), (24.865, 0.0), (5.8269, 0.0), (8.9294, 0.0009)] (Avg: (9.6841, 0.1950))
LID size: 6179.0000.


  0%|▏                                                                                | 1/500 [00:03<31:45,  3.82s/it, loss=0.3943, acc=0.9141, size=853.26]


LID size: 853.2612 with certificate of 0.9140625.


...

Test Results: [(0.1775, 0.9721), (7.5932, 0.0005), (24.6393, 0.0), (6.4401, 0.0), (8.9819, 0.0009)] (Avg: (9.5664, 0.1947))
Target acc == 0, no need to recompute LID.


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 15/15 [00:41<00:00,  2.74s/it, val_loss=6.5128, val_acc=0.0000]


Test Results: [(0.1777, 0.9721), (7.6665, 0.0005), (25.1602, 0.0), (6.1292, 0.0005), (9.0328, 0.0005)] (Avg: (9.6333, 0.1947))
Target acc == 0, no need to recompute LID.


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 15/15 [00:43<00:00,  2.88s/it, val_loss=8.3094, val_acc=0.0008]


Test Results: [(0.1773, 0.9711), (7.7316, 0.0005), (25.675, 0.0), (6.2859, 0.0), (8.7392, 0.0009)] (Avg: (9.7218, 0.1945))


### Class Incremental Learning + Regulariser

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_fully_connected_mnist_model(device="cuda", hidden_dim=400)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
unbias = UnbiasRegulariser(lmbd=0)
l2 = L2Regulariser(lmbd=1)
regulariser = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=1, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1], regulariser=regulariser)

In [ ]:
interval_trainer = InterContiNetTrainer(
    trainer.model,
    min_acc_limit=0.9,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256)
    test_vals = interval_trainer.test(test_tasks[0 : i + 1])
    if not test_vals[-1][1]:
        break
    if i < len(train_tasks) - 2:
        target_acc = min(
            max(test_vals[-1][1] - interval_trainer.min_acc_increment, 0),
            interval_trainer.min_acc_limit,
        )
        interval_trainer.min_acc_limit = target_acc
        interval_trainer.compute_rashomon_set(test)

## CIFAR

In [5]:
import torchvision
from src.data_utils import split_mnist_by_labels
import copy
from src.utils.general import set_seed

### Domain Incremental Learning

In [6]:
device = "cuda"
classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

task_pairs = [("cat", "truck"), ("frog", "ship"), ("horse", "automobile"), ("dog", "airplane"), ("bird", "deer")]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

def domain_map_fn(y):
    print("in:", y)
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1 ,0, 0]).to(device)
    print("out:", animals_mask[y])
    return animals_mask[y]

@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset

def get_tasks(encoder, seed: int = 42):
    set_seed(seed)
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)]
    print("Contexts:",  task_pairs_ids)    
    trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids]
    train_tasks = [split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids]

    return train_tasks, test_tasks

In [7]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=transform)
cifar100_testset = torchvision.datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [8]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")
def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model

model = load_pretrained_model(model_path)

In [9]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [16]:
def domain_map_fn(y):
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1 ,0, 0]).to(device)
    return animals_mask[y]

SEED = 2
train_tasks, test_tasks = get_tasks(encoder, seed=SEED)
model = models.get_fully_connected_model(
    device="cuda",
    output_dim=10,
    input_dim=512,
    seed=SEED,
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model,
    min_acc_increment=0.2,
    min_acc_limit=1,
    paradigm="CIL",
    seed=SEED,
)

lr = 0.01
batch_size = 128
epochs = 30
lid_lr = 0.1
for i, (train, test) in enumerate(zip(train_tasks, test_tasks)):
    interval_trainer.train(
        train,
        test,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    results = interval_trainer.test(test_tasks)
    if i == len(train_tasks) - 1:
        continue
    target_acc = max(results[i][1] - interval_trainer.min_acc_increment, results[i][1] / 2)
    interval_trainer.min_acc_limit = target_acc
    print("target acc:", target_acc)
    interval_trainer.compute_rashomon_set(test, lr=lid_lr, default_eps=0.1)

Contexts: [[3, 0], [4, 1], [5, 9], [7, 8], [2, 6]]


/tmp/ipykernel_27116/904684961.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_27116/904684961.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Tasks: [[0, 3], [1, 4], [5, 9], [7, 8], [2, 6]]


Training Epochs: 100%|█████████████████████████████████████████████████████████████████████| 30/30 [01:07<00:00,  2.25s/it, val_loss=0.3438, val_acc=0.8665]


Test Results: [(0.3595, 0.8695), (96.0624, 0.0), (102.921, 0.0), (93.2327, 0.0), (91.7869, 0.0)] (Avg: (76.8725, 0.1739))
target acc: 0.6695
LID size: 5130.0000.


...

Test Results: [(0.3493, 0.8745), (84.2475, 0.0), (89.8318, 0.0005), (73.9928, 0.0005), (101.5913, 0.0)] (Avg: (70.0025, 0.1751))
target acc: 0.00025
LID size: 161.1604.


  1%|▍                                                                             | 3/500 [00:03<08:53,  1.07s/it, loss=74.8738, acc=0.007812, size=109.60]


LID size: 109.5975 with certificate of 0.0078125.


Training Epochs: 100%|████████████████████████████████████████████████████████████████████| 30/30 [01:09<00:00,  2.30s/it, val_loss=82.3447, val_acc=0.0005]


Test Results: [(0.3477, 0.8745), (90.3158, 0.0), (95.144, 0.0005), (76.4614, 0.0005), (82.2614, 0.0005)] (Avg: (68.9061, 0.1752))
